In [1]:
from cgra import *
from kernels import *
from sat_to_csv import *

In [2]:
kernel_name = "gemm"
version = "_meth_lwd"

In [3]:
# Global variables
CGRA_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr = 20000

In [4]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [5]:
# Data
def configMemory(A_data, B_data, C_data, rowsA, colsA, colsB, alpha, beta):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    # &B[0][0]          &C[0][1]        &A[0][0]        nRowsBlocksC
    # nColsBlocksC      &B[0][1]        &C[1][2]        &A[1][0]
    # &A[2][0]          loopColsA       &B[0][2]        &C[2][3]
    # &C[3][0]          &A[3][0]        nColsBlocksC    &B[0][3]
    # ----------------------
    # -4*colsB          colsA           alpha           -
    # -                 -4*colsB        colsA           alpha
    # beta              -               -4*colsB        colsA
    # colsA             beta            -               -4*colsB
    nItLoopColsA = colsA
    nColsBlocksC = int(colsB/CGRA_N_ROWS)
    nRowsBlocksC = int(rowsA/CGRA_N_ROWS)
    first_addr_A = first_addr
    first_addr_B = first_addr_A + rowsA*colsA*4
    first_addr_C = first_addr_B + colsA*colsB*4
    config_vals_col0 = [first_addr_B, nColsBlocksC, first_addr_A + 2*colsA*4, first_addr_C + 3*colsB*4, -4*colsB, beta, colsA]
    config_vals_col1 = [first_addr_C + 4, first_addr_B + 4, nItLoopColsA, first_addr_A + 3*colsA*4, colsA, -4*colsB, beta]
    config_vals_col2 = [first_addr_A, first_addr_C + 2*4 + colsB*4, first_addr_B + 2*4, nColsBlocksC, alpha, colsA, -4*colsB]
    config_vals_col3 = [nRowsBlocksC, first_addr_A + colsA*4, first_addr_C + 3*4 + 2*colsB*4, first_addr_B + 3*4, alpha, colsA, -4*colsB]
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_B, B_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_C, C_data, version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3]
    return load_addrs

In [6]:
def runKernel(load_addrs, max_it=1000):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [7]:
def getResult(first_addr_C, end_addr_C, rowsA, colsB):
    result = [0 for _ in range(rowsA*colsB)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [8]:
def gemm_cpu(A_data, B_data, C_data, rowsA, colsA, colsB, alpha, beta):
    expected_res = [0 for _ in range(rowsA*colsB)]
    for rA in range(rowsA):
        for cB in range(colsB):
            sum = 0
            for cA in range(colsA):
                sum += A_data[rA*colsA + cA] * B_data[cA*colsB + cB]
            expected_res[rA*colsB + cB] = alpha * sum + beta * C_data[rA*colsB + cB]
    return expected_res

In [9]:
# Test dimensions (4xXx4)
rowsA = 8
colsA = 8
colsB = 8
A_data = list(range(0, rowsA * colsA))
B_data = [x + 100 for x in range(0, colsA * colsB)]
C_data = [x + 200 for x in range(0, rowsA * colsB)]

A_data_cpy = A_data.copy()
B_data_cpy = B_data.copy()
C_data_cpy = C_data.copy()

alpha = 125 
beta = 14
#print("A")
#printAsMatrix(A_data, rowsA, colsA)
#print("B")
#printAsMatrix(B_data, colsA, colsB)
load_addrs = configMemory(A_data, B_data, C_data, rowsA, colsA, colsB, alpha, beta)

In [10]:
runKernel(load_addrs, max_it=200000)

Instr =  0 ( 0 )
[20256, 20516, 20000,    2]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4]    
[   2, 20260, 20552, 20032]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4]    
[20064,    8, 20264, 20588]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4]    
[20608, 20096,    2, 20268]    [LWD R2  4, LWD R2  4, LWD R2  4, LWD R2  4]    
-------
Instr =  1 ( 1 )
[ -32,    8,  125,    0]    [LWD R1  4, LWD R1  4, LWD R1  4, SADD R1  ZERO  ZERO]    
[   0,  -32,    8,  125]    [SADD R1  ZERO  ZERO, LWD R1  4, LWD R1  4, LWD R1  4]    
[  14,    0,  -32,    8]    [LWD R1  4, SADD R1  ZERO  ZERO, LWD R1  4, LWD R1  4]    
[   8,   14,    2,  -32]    [LWD R1  4, LWD R1  4, NOP , LWD R1  4]    
-------
Instr =  2 ( 2 )
[   0,    0,    0,    0]    [SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO]    
[   0,    0,    0,    0]    [SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO, SADD R3  ZERO  ZERO]    
[   0,    0,    0,    0]    [SADD R3  ZERO  ZE

In [12]:
# Get result from CGRA
first_addr_C = first_addr + rowsA*colsA*4 + colsA*colsB*4
result = getResult(first_addr_C, first_addr_C + rowsA*colsB*4, rowsA, colsB)
# Process estra rows/cols
if rowsA%4 != 0:
    for rA in range(rowsA - rowsA%4, rowsA):
        for cB in range(colsB):
            for k in range(colsA):
                result[rA*colsB+cB] += A_data[rA*colsA+k]*B_data[k*colsB+cB]
if colsB%4 != 0:
    for cB in range(colsB - colsB%4, colsB):
        for rA in range(rowsA - rowsA%4):
            for k in range(colsA):
                result[rA*colsB+cB] += A_data[rA*colsA+k]*B_data[k*colsB+cB]

# Get cpu output
expected_res = gemm_cpu(A_data_cpy, B_data_cpy, C_data_cpy, rowsA, colsA, colsB, alpha, beta)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    printAsMatrix(expected_res, rowsA, colsB)
    printAsMatrix(result, rowsA, colsB)
else:
    print("OK")



Err: 16
[492800, 496314, 499828, 503342, 506856, 510370, 513884, 517398]
[1516912, 1528426, 1539940, 1551454, 1562968, 1574482, 1585996, 1597510]
[2541024, 2560538, 2580052, 2599566, 2619080, 2638594, 2658108, 2677622]
[3565136, 3592650, 3620164, 3647678, 3675192, 3702706, 3730220, 3757734]
[4589248, 4624762, 4660276, 4695790, 4731304, 4766818, 4802332, 4837846]
[5613360, 5656874, 5700388, 5743902, 5787416, 5830930, 5874444, 5917958]
[6637472, 6688986, 6740500, 6792014, 6843528, 6895042, 6946556, 6998070]
[7661584, 7721098, 7780612, 7840126, 7899640, 7959154, 8018668, 8078182]
[492800, 495754, 499828, 503342, 506856, 7448056, 513884, 517398]
[1516912, 1528426, 1539884, 1551454, 1562968, 1574482, 23141376, 1597510]
[2541024, 2560538, 2580052, 2599510, 2619080, 2638594, 2658108, 39067640]
[3565080, 3592650, 3620164, 3647678, 53583120, 3702706, 3730220, 3757734]
[4589248, 56459384, 4660276, 4695790, 4731304, 795194876, 4802332, 4837846]
[5613360, 5656874, 5700332, 5743902, 5787416, 583093